In [ ]:
%pip install diffusers transformers accelerate torch torchvision Pillow peft controlnet_aux

# 2. Inference

In [ ]:
import os
import json
import torch
from pathlib import Path
from PIL import Image
from peft import PeftModel
from diffusers import StableDiffusionImg2ImgPipeline, UniPCMultistepScheduler, StableDiffusionControlNetImg2ImgPipeline, ControlNetModel
from controlnet_aux import CannyDetector
from diffusers.utils import load_image, make_image_grid




In [ ]:
def resize_img(img: Image.Image, max_side: int = 768, multiple_of: int = 8):
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    new_w, new_h = round(w * scale / multiple_of) * multiple_of, round(h * scale / multiple_of) * multiple_of
    if (new_w, new_h) == (w, h):
        return img

    img = img.resize((new_w, new_h), Image.LANCZOS)
    return img

In [ ]:
def create_inference_pipeline(
                          lora_weights,
                          device,
                          model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5",
                          controlnet_id = "lllyasviel/sd-controlnet-canny"):

    controlnet = ControlNetModel.from_pretrained(
        controlnet_id,
        torch_dtype=torch.float16,
        use_safetensors=True
    )

    pipeline = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        model_id,
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(device)

    pipeline.scheduler = UniPCMultistepScheduler.from_config(pipeline.scheduler.config)

    # patch missing base_model_name_or_path in config
    config_path = os.path.join(lora_weights, "adapter_config.json")
    with open(config_path) as f:
        config = json.load(f)
    if config.get("base_model_name_or_path") is None:
        config["base_model_name_or_path"] = model_id
        with open(config_path, "w") as f:
            json.dump(config, f, indent=2)

    pipeline.unet = pipeline.unet.to(torch.float32)
    pipeline.unet = PeftModel.from_pretrained(pipeline.unet, lora_weights, is_trainable=False)
    pipeline.unet = pipeline.unet.merge_and_unload()
    pipeline.unet = pipeline.unet.to(torch.float16).to(device)
    return pipeline

In [ ]:
LORA_WEIGHTS_DIR = "model_400"
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = create_inference_pipeline(LORA_WEIGHTS_DIR, DEVICE)



In [ ]:
SEED = 42
generator = torch.Generator(device="cpu").manual_seed(SEED)

In [ ]:
OUTPUT_DIR = "output_data"
STYLE_TYPE = "van Gogh"
NEG_PROMPT = "low quality, blurry, distorted, deformed, ugly"

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number = 2):
  content_img = resize_img(load_image(content_path))
  canny = CannyDetector()
  edge_img = canny(content_img, detect_resolution=min(content_img.size), image_resolution=min(content_img.size))
  stylised_images = pipeline(
      prompt=input_prompt,
      negative_prompt=NEG_PROMPT,
      image=content_img,
      control_image=edge_img,
      strength=strength,
      guidance_scale=guidance_scale,
      controlnet_conditioning_scale=control_net_scale,
      num_images_per_prompt=images_number,
      generator=generator,
  ).images

  return [content_img] + stylised_images

In [ ]:
input_prompt   = f"stylization in {STYLE_TYPE} style"
strength = 0.9
guidance_scale = 8
control_net_scale = 0.7
images_number = 2

### Puppy

In [ ]:
content_path = "01_content/1.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[1].save(OUTPUT_DIR+"/1_lora.jpg", quality=95)

### Thailand

In [ ]:
content_path = "01_content/2.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[2].save(OUTPUT_DIR+"/2_lora.jpg", quality=95)

### Flower field

In [ ]:
content_path = "01_content/3.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[2].save(OUTPUT_DIR+"/3_lora.jpg", quality=95)

### Girl

In [ ]:
content_path = "01_content/4.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[1].save(OUTPUT_DIR+"/4_lora.jpg", quality=95)

### Venice

In [ ]:
content_path = "01_content/5.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[1].save(OUTPUT_DIR+"/5_lora.jpg", quality=95)

### Dog

In [ ]:
content_path = "01_content/6.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[1].save(OUTPUT_DIR+"/6_lora.jpg", quality=95)

### Paris

In [ ]:
content_path = "01_content/7.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[1].save(OUTPUT_DIR+"/7_lora.jpg", quality=95)

### Italy

In [ ]:
content_path = "01_content/8.jpg"
output_images = stylise(input_prompt, content_path, strength, guidance_scale, control_net_scale, images_number)
make_image_grid(output_images, cols=images_number+1, rows=1)

In [ ]:
output_images[1].save(OUTPUT_DIR+"/8_lora.jpg", quality=95)